In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [8]:
import tensorflow as tf
import keras
import zipfile
import json

model_path = "/content/drive/MyDrive/NLP/models/Embedding_CNN.keras"

print("Current TensorFlow:", tf.__version__)
print("Current Keras:", keras.__version__)

with zipfile.ZipFile(model_path) as archive:
    print("Saved metadata:")
    print(json.loads(archive.read("metadata.json")))

Current TensorFlow: 2.21.0
Current Keras: 3.13.2
Saved metadata:
{'keras_version': '3.13.2', 'date_saved': '2026-11-04@13:26:35'}


In [9]:
import tensorflow as tf

model_path = "/content/drive/MyDrive/NLP/models/Embedding_CNN.keras"
loaded_model = tf.keras.models.load_model(model_path, compile=False)

# Verify that the interface receives five star probabilities per review.
probs = loaded_model(tf.constant(["Produto muito bom"]), training=False)
print("Output shape:", probs.shape)  # Expected: (1, 5)

Output shape: (1, 5)


In [10]:
import tensorflow as tf
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

title_input = widgets.Text(
    description="Título:",
    placeholder="Título opcional",
    layout=widgets.Layout(width="600px")
)

comment_input = widgets.Textarea(
    description="Comentário:",
    placeholder="Escreva seu comentário em português...",
    layout=widgets.Layout(width="600px", height="120px")
)

predict_button = widgets.Button(
    description="Prever nota",
    button_style="primary",
    icon="star"
)

output = widgets.Output()

def predict_rating(_):
    with output:
        clear_output(wait=True)

        text = f"{title_input.value} {comment_input.value}".strip()

        if not text:
            print("Por favor, escreva um título ou comentário.")
            return

        inputs = tf.constant([text])

        if hasattr(loaded_model, "serve"):
            result = loaded_model.serve(inputs)
        else:
            result = loaded_model(inputs, training=False)

        probabilities = result.numpy()[0]
        score = int(np.argmax(probabilities)) + 1

        print(f"Nota prevista: {score}/5")
        print("★" * score + "☆" * (5 - score))

        print("\nProbabilidades estimadas:")
        for star, probability in enumerate(probabilities, start=1):
            print(f"  Nota {star}: {probability:.1%}")

predict_button.on_click(predict_rating)

display(widgets.VBox([
    title_input,
    comment_input,
    predict_button,
    output
]))